# TPCx-AI UC9 — Databricks Data Load

Loads CUSTOMER_IMAGES into Unity Catalog managed volume, ready for distributed PyTorch training notebook. Images are copied from S3 to managed volumes.

## Prerequisites

1. Generate UC9 images with the official [TPCx-AI kit](https://www.tpc.org/tpcx-ai/) (PDGF). You need the `CUSTOMER_IMAGES/` directory with subdirectories per identity (ImageFolder format):
   ```
   CUSTOMER_IMAGES/
     ├── identity_001/
     │   ├── sample_001.png
     │   ├── sample_002.png
     │   └── ...
     ├── identity_002/
     └── ...
   ```

2. Upload to your S3 bucket under `s3a://YOUR_BUCKET/tpcxai/uc9/CUSTOMER_IMAGES/`

3. Replace `YOUR_BUCKET`, `YOUR_AWS_KEY_ID`, `YOUR_AWS_SECRET_KEY` in the configuration cells below. For production, prefer an instance profile or Unity Catalog external location over inline keys.

## Running

1. Attach this notebook to a cluster running Databricks Runtime.
2. Run all cells.

## Data Flow

```
S3 External Storage                     Unity Catalog Managed Volume
s3a://bucket/tpcxai/uc9/           →    /Volumes/tpcxai_uc9/public/uc9_images/
  CUSTOMER_IMAGES/                       CUSTOMER_IMAGES/
    ├── identity_001/                      ├── identity_001/
    │   ├── 1.png                         │   ├── 1.png
    │   └── 2.png                         │   └── 2.png
    └── identity_002/                      └── identity_002/
```

In [ ]:
# S3 credentials (one-time per cluster session). Replace placeholders before running.
spark.conf.set("fs.s3a.access.key", "YOUR_AWS_KEY_ID")
spark.conf.set("fs.s3a.secret.key", "YOUR_AWS_SECRET_KEY")

# Sanity-check the bucket is reachable
dbutils.fs.ls("s3a://YOUR_BUCKET/")

In [ ]:
# Configuration
S3_BASE = "s3a://YOUR_BUCKET/tpcxai/uc9"
CATALOG = "tpcxai_uc9"  # Unity Catalog catalog name
SCHEMA = "public"  # Schema name

print(f"S3 base path: {S3_BASE}")
print(f"Target catalog: {CATALOG}")
print(f"Schema: {SCHEMA}")

In [ ]:
# Create Unity Catalog catalog and schema (one-time)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print(f"Catalog '{CATALOG}' ready")
print(f"Schema '{CATALOG}.{SCHEMA}' ready")

In [ ]:
from datetime import datetime

print(f"\n{'='*60}")
print(f"Loading UC9 Images")
print(f"{'='*60}")

# Create managed volume
volume = "uc9_images"
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{volume}")
print(f"Volume '{CATALOG}.{SCHEMA}.{volume}' ready")

# Paths
s3_source = f"{S3_BASE}/CUSTOMER_IMAGES"
volume_dest = f"/Volumes/{CATALOG}/{SCHEMA}/{volume}/CUSTOMER_IMAGES"

print(f"\nCopying images...")
print(f"  Source: {s3_source}")
print(f"  Destination: {volume_dest}")

# Check if data already exists
try:
    existing_files = dbutils.fs.ls(volume_dest)
    print(f"\nVolume already contains {len(existing_files)} items")
    print(f"   Removing existing data...")
    dbutils.fs.rm(volume_dest, recurse=True)
except:
    pass  # Volume is empty, proceed

# Copy from S3 to Volume
start = datetime.now()
dbutils.fs.cp(s3_source, volume_dest, recurse=True)
elapsed = (datetime.now() - start).total_seconds()

print(f"\nCopy complete in {elapsed:.1f}s")

# Verify and count
print(f"\nVerifying data...")
identities = [f.name for f in dbutils.fs.ls(volume_dest) if f.isDir()]
total_images = 0
for identity in identities:
    identity_path = f"{volume_dest}/{identity}"
    images = [f for f in dbutils.fs.ls(identity_path) if f.name.endswith('.png')]
    total_images += len(images)

print(f"\nVerification complete")
print(f"  Total identities: {len(identities)}")
print(f"  Total images: {total_images:,}")
print(f"  First 5 identities: {identities[:5]}")

print(f"\n{'='*60}")
print("Done! Images loaded into managed volume.")
print(f"{'='*60}")
print(f"\nUse in training notebook:")
print(f"  IMAGE_ROOT = '{volume_dest}'")

## Summary

Images are now loaded into managed volume and ready for training.

To use in `uc9_pytorch_benchmark_databricks.ipynb`:

```python
IMAGE_ROOT = "/Volumes/tpcxai_uc9/public/uc9_images/CUSTOMER_IMAGES/"
```